# AML - SAIGE-GENE SET-BASED ANALYSIS - NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [ ]:
# Load packages 
import os

In [ ]:
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')
# Set environment variable for workspace bucket 
my_bucket


# COMMAND 

In [ ]:
%%writefile saige_gene_step01_deprivation_index.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_phenotype; do 
    echo "Processing phenotype: ${pheno}"
    
    step1_fitNULLGLMM.R \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
        --phenoFile="${phenotype}/${pop}_deprivation_index_file.txt" \
        --useSparseGRMtoFitNULL=TRUE \
        --useSparseGRMforVarRatio=TRUE \
        --IsOverwriteVarianceRatioFile=TRUE \
        --isCateVarianceRatio=TRUE \
        --phenoCol="${pheno}" \
        --traitType=binary \
        --covarColList=sex,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,deprivation_index \
        --qCovarColList=sex \
        --sampleIDColinphenoFile=person_id \
        --outputPrefix="${output}/${pop}_${pheno}_deprivation_index_null_glmm" 
    done
done


In [ ]:
# Save script to workspace bucket 
!gsutil cp saige_gene_step01_deprivation_index.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/saige_gene_step01_deprivation_index.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 200 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/saige_gene_step01_deprivation_index.sh"
